# Решения: практика буферов

**Для преподавателя.** Полный эталон к `lesson.ipynb` и `homework.ipynb`; ученикам до сдачи не показывать.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def find_orders_csv() -> Path:
    for path in (Path("orders_slim.csv"), Path("../../data/orders_slim.csv")):
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(
        "orders_slim.csv не найден рядом с ноутбуком или в ../../data/"
    )


CSV_PATH = find_orders_csv()
DATE_COLUMNS = [
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
    "order_delivered_customer_date",
]
df = pd.read_csv(CSV_PATH, parse_dates=DATE_COLUMNS)
assert len(df) > 0
assert df["order_id"].notna().all()
print(f"Загружено заказов: {len(df)}")

from collections import deque


## Урок. 1. Последние события

In [ ]:
recent = deque(maxlen=7)
for oid in df["order_id"]:
    recent.append(oid)
assert list(recent) == df["order_id"].tail(min(7, len(df))).tolist()


## Урок. 2. Скользящее среднее

In [ ]:
def rolling_mean(values, width):
    window, result = deque(maxlen=width), []
    for value in values:
        window.append(float(value))
        if len(window) == width:
            result.append(float(sum(window) / width))
    return result

means5 = rolling_mean(df["delivery_days"].tolist(), 5)
assert len(means5) == max(0, len(df) - 4)


## Урок. 3–5. Очереди и квота

In [ ]:
late_q, normal_q = deque(), deque()
for row in df[["order_id", "is_late"]].itertuples(index=False):
    (late_q if row.is_late else normal_q).append(row.order_id)
processed = []
while len(processed) < min(12, len(df)) and (late_q or normal_q):
    for _ in range(3):
        if late_q and len(processed) < min(12, len(df)):
            processed.append((late_q.popleft(), "late"))
    if normal_q and len(processed) < min(12, len(df)):
        processed.append((normal_q.popleft(), "normal"))
remaining = len(late_q) + len(normal_q)
assert len(processed) + remaining == len(df)


## Урок. 6. Отмена разметки

In [ ]:
labels = [(oid, "checked") for oid in df["order_id"].head(6)]
reverted = [labels.pop(), labels.pop()]
assert len(labels) == 4 and len(reverted) == 2


## Урок. 7. Соседние повторы

In [ ]:
raw = ["A", "A", "B", "B", "A", "C", "C"]
buffer = deque()
for event in raw:
    if not buffer or buffer[-1] != event:
        buffer.append(event)
compact = list(buffer)
assert compact == ["A", "B", "A", "C"]


## Урок. 8. Эксперимент квот

In [ ]:
def group_order(frame, quota, limit):
    lq = deque(frame.loc[frame["is_late"].eq(1), "order_id"])
    nq = deque(frame.loc[frame["is_late"].eq(0), "order_id"])
    groups = []
    while len(groups) < limit and (lq or nq):
        for _ in range(quota):
            if lq and len(groups) < limit:
                lq.popleft(); groups.append("late")
        if nq and len(groups) < limit:
            nq.popleft(); groups.append("normal")
        if not lq:
            while nq and len(groups) < limit:
                nq.popleft(); groups.append("normal")
    return groups

orders_by_quota = {q: group_order(df, q, min(16, len(df))) for q in (1, 3)}
QUOTA_NOTE = "Квота 3 быстрее уменьшает очередь late, но normal ждут дольше. Квота 1 даёт более ровное обслуживание, хотя критичный хвост сокращается медленнее."
assert len(QUOTA_NOTE) >= 100


## Урок. 9. Процессор

In [ ]:
def process_stream(frame, limit):
    lq = deque(frame.loc[frame["is_late"].eq(1), "order_id"])
    nq = deque(frame.loc[frame["is_late"].eq(0), "order_id"])
    done = []
    while len(done) < limit and (lq or nq):
        done.append((lq if lq else nq).popleft())
    return done, len(lq), len(nq)

done, late_left, normal_left = process_stream(df, min(15, len(df)))
assert len(done) + late_left + normal_left == len(df)


## ДЗ. A1–A2. Окна

In [ ]:
freight_means = rolling_mean(df["freight_value"].tolist(), 4)
window = deque(maxlen=8)
window_max = []
for value in df["delay_days"]:
    window.append(float(value))
    if len(window) == 8:
        window_max.append(max(window))
assert len(window_max) == max(0, len(df) - 7)


## ДЗ. A3. Чередование

In [ ]:
lq = deque(df.loc[df["is_late"].eq(1), "order_id"])
nq = deque(df.loc[df["is_late"].eq(0), "order_id"])
alternating = []
while lq and nq:
    alternating.extend([(lq.popleft(), "late"), (nq.popleft(), "normal")])
assert all(alternating[i][1] != alternating[i + 1][1] for i in range(len(alternating) - 1))


## ДЗ. Challenge

In [ ]:
def adaptive_dispatch(frame, limit, threshold):
    lq = deque(frame.loc[frame["is_late"].eq(1), "order_id"])
    nq = deque(frame.loc[frame["is_late"].eq(0), "order_id"])
    result, turn_late = [], True
    while len(result) < limit and (lq or nq):
        if lq and (len(lq) > threshold or turn_late or not nq):
            result.append(lq.popleft())
        else:
            result.append(nq.popleft())
        turn_late = not turn_late
    return result

adaptive = adaptive_dispatch(df, min(20, len(df)), 3)
BUFFER_NOTE = (
    "Deque даёт O(1) для снятия события слева. При постоянном приоритете late очередь normal может голодать. "
    "Защита — квота или чередование: после нескольких late обязательно обработать normal, сохранив FIFO внутри групп."
)
assert len(adaptive) == min(20, len(df)) and len(BUFFER_NOTE) >= 180
